# PD Analysis (Modular)

This notebook is a concise entry point. All heavy lifting lives in `pd_pipeline/`.


In [ ]:
import pandas as pd

from pd_pipeline import basel, capital, config, data, lasso, plots, portfolio, scenario, sensitivity


In [ ]:
# Load and merge macro + GPR data
macro_frames = data.load_macro_data(
    gdp_path='data/macro/sweden_gdp_monthly.csv',
    interest_path='data/macro/sweden_interest_rate_monthly.csv',
    unemployment_path='data/macro/sweden_unemployment_monthly.csv',
    housing_path='data/macro/Bostadspriser ScB 1997-2025_transposed.csv',
    cpi_path='data/macro/KPI SCB 1980-2026.csv',
    verbose=True,
)

df_gpr = data.load_gpr_data('data/geopolitical/data_gpr_Data_GPR.csv', verbose=True)

df_merged = data.merge_macro_data(macro_frames, df_gpr)

cov_matrix, corr_matrix, mean_vector = data.summarize_macro_data(
    df_merged,
    config.ALL_PREDICTOR_COLS,
    verbose=True,
)


In [ ]:
# Load PDs and merge with macro data

# Note: filesystem is case-insensitive on macOS, so data/PDs and data/pds both work

df_pds = data.load_pds_data('data/PDs/pdsFitchData.csv', verbose=True)

df_final = data.merge_pds_macro(df_pds, df_merged, verbose=True)

df_final_cleaned = data.prepare_model_data(
    df_final,
    config.ALL_PREDICTOR_COLS,
    sector_col=config.SECTOR_COL,
    verbose=True,
)


In [ ]:
# Export cleaned dataset for reuse
data.export_dataframe(df_final_cleaned, output_file='df_final_cleaned.csv', verbose=True)


In [ ]:
# OLS sensitivity analysis

df_sensitivities = sensitivity.run_sensitivity_analysis(
    df_final_cleaned,
    macro_cols=config.MACRO_COLS,
    gpr_cols=config.GPR_COLS,
    sector_col=config.SECTOR_COL,
    pd_maturity_cols=config.PD_MATURITY_COLS,
    pdzero_col=config.PDZERO_COL,
    verbose=True,
)

print("\n" + "="*80)
print("SENSITIVITY ANALYSIS RESULTS")
print("="*80)
print(df_sensitivities)


In [ ]:
# Sensitivity exports + tables
sensitivity.export_sensitivities(df_sensitivities, output_file='sensitivity_results_with_CI.csv')

sensitivity.print_sensitivity_tables(df_sensitivities, config.MACRO_COLS, config.GPR_COLS)

sensitivity.print_confidence_interval_summary(df_sensitivities, config.GPR_COLS)

# Uncomment for a full per-sector printout (very verbose)
# sensitivity.print_sensitivity_details(df_sensitivities, config.MACRO_COLS, config.GPR_COLS)


In [ ]:
# LASSO feature selection

df_lasso, lasso_selected_features = lasso.run_lasso_feature_selection(
    df_final_cleaned,
    macro_cols=config.MACRO_COLS,
    gpr_cols=config.GPR_COLS,
    sector_col=config.SECTOR_COL,
    pd_maturity_cols=config.PD_MATURITY_COLS,
    pdzero_col=config.PDZERO_COL,
    verbose=True,
)

feature_freq_df = lasso.print_lasso_summary(df_lasso, config.MACRO_COLS, config.GPR_COLS)
comparison_full = lasso.compare_ols_lasso(df_sensitivities, df_lasso, config.MACRO_COLS, config.GPR_COLS)
lasso.export_lasso_outputs(df_lasso, comparison_full)
lasso.print_feature_recommendations(feature_freq_df, comparison_full)


In [ ]:
# LASSO visualizations
plots.plot_lasso_summary(df_lasso, feature_freq_df, config.MACRO_COLS, config.GPR_COLS, df_sensitivities)


In [ ]:
# Sensitivity regression plots
plots.plot_sector_regressions(df_sensitivities, df_final_cleaned, config.MACRO_COLS, config.GPR_COLS, config.SECTOR_COL)

plots.plot_sector_comparison(
    df_sensitivities,
    df_final_cleaned,
    config.MACRO_COLS,
    sector_col=config.SECTOR_COL,
    title='Regression Lines for All Sectors - Macro Variables',
    ylabel='Δ logit(PD)',
)


In [ ]:
# Basel asset correlations (create/refresh correlation file)

df_exposures = pd.read_csv('data/PDs/pdsFitchData_latest_exposures.csv')
print(f"Loaded {len(df_exposures)} exposures from {df_exposures['Date'].iloc[0]}")
print(f"Unique companies: {df_exposures['Company_number'].nunique()}")
print(f"\nColumns: {df_exposures.columns.tolist()}")

print("\nExample: PD = 8.33% → ρ = {:.4f}".format(basel.asset_correlation_formula(0.0833)))

pd_columns = config.DEFAULT_PD_TENORS

df_exposures = basel.append_basel_correlations(df_exposures, pd_columns, verbose=True)

sample_cols = ['Company_number', 'Sector', '12_month', '12_month_correlation']
print("Sample results (12-month tenor):")
print("\n" + df_exposures[sample_cols].dropna(subset=['12_month']).head(15).to_string(index=False))

# Save for downstream steps
output_corr_path = 'data/PDs/pdsFitchData_latest_with_basel_correlation.csv'
df_exposures.to_csv(output_corr_path, index=False)
print(f"\n✓ Correlation-augmented exposures saved to: {output_corr_path}")


In [ ]:
# Scenario-based portfolio loss simulation
mean_vec = mean_vector[config.ALL_PREDICTOR_COLS].values
cov_mat = cov_matrix.loc[config.ALL_PREDICTOR_COLS, config.ALL_PREDICTOR_COLS].values

scenario_results = scenario.calculate_scenario_portfolio_loss(
    exposures_csv='data/PDs/pdsFitchData_latest_with_basel_correlation.csv',
    sensitivities_csv='sensitivity_results_with_CI.csv',
    macro_vars=config.MACRO_COLS,
    gpr_vars=config.GPR_COLS,
    n_scenarios=10000,
    tenor='12_month',
    ead=1_000_000,
    lgd=0.45,
    quantile=0.999,
    seed=42,
    mean_vec=mean_vec,
    cov_mat=cov_mat,
    verbose=True,
)


In [ ]:
# Scenario loss plots
plots.plot_scenario_loss(scenario_results)


In [ ]:
# Deterministic portfolio loss (single-quantile) + plots
portfolio_results = portfolio.calculate_portfolio_loss(
    exposures_csv='data/PDs/pdsFitchData_latest_with_basel_correlation.csv',
    tenor='12_month',
    ead=1_000_000,
    lgd=0.45,
    quantile=0.999,
    verbose=True,
)

plots.plot_portfolio_loss_breakdown(portfolio_results)


In [ ]:
# RWA calculations
print("=" * 70)
print("CALCULATING RWA FOR ENTIRE PORTFOLIO")
print("=" * 70)

# Load the data with Basel correlations
df_portfolio = pd.read_csv('data/PDs/pdsFitchData_latest_with_basel_correlation.csv')

print(f"\nDataset loaded: {len(df_portfolio)} exposures")
print(f"Columns: {list(df_portfolio.columns)}")

# Parameters
LGD = 0.45
EAD = 1_000_000
MATURITY = 2.5

print(f"\nPortfolio parameters:")
print(f"  LGD: {LGD:.2%}")
print(f"  EAD per exposure: {EAD:,.0f} SEK")
print(f"  Maturity: {MATURITY} years")
print(f"  Total Portfolio EAD: {EAD * len(df_portfolio):,.0f} SEK")

results_by_tenor = basel.compute_rwa_by_tenor(
    df_portfolio,
    tenors=config.DEFAULT_RWA_TENORS,
    lgd=LGD,
    ead=EAD,
    maturity=MATURITY,
)

basel.print_rwa_summary(results_by_tenor)
basel.print_rwa_detail_12m(results_by_tenor)


In [ ]:
# CET1 ratio analysis
RWA_0 = 540_000_000
CET1_0 = RWA_0 * 0.18

cet1_results = capital.cet1_analysis(
    rwa_0=RWA_0,
    cet1_0=CET1_0,
    portfolio_loss=portfolio_results['total_loss'],
    results_by_tenor=results_by_tenor,
    tenor='12_month',
    verbose=True,
)
